# Seiler oscillator
<figure>
<img src="../VFO/doc/EMRFD_fig.4.11.png" width="600px" />
<figcaption>EMRFD Fig. 4.11</figcaption>
</figure>

## Transistor biasing
1. The original 8V implementation has a 1mA emitter current.
2. The collector voltage is about halfway between the supply and ground (at 4.46V).
3. The emitter voltage is about halfway between the collector and ground (at 2.20V).

In [5]:
# The reference designators used in the calculations are the ones in the KiCad schematic.

Vcc = 5 # Supply voltage
Ie_Q1 = 1e-3 # Emitter current of Q1
Ib_Q1 = Ie_Q1 / 100 # Base current of Q1 (assuming beta = 100)
I_R2 = 10 * Ib_Q1 # Current through R2 (10 times the base current of Q1 for good biasing)
I_R3 = I_R2 + Ie_Q1 # Current through R3 (sum of current through R2 and emitter current of Q1, assuming negligible base current of Q2)
V_R3 = Vcc / 2 # Voltage across R3 (half of the supply voltage for proper biasing)
R3 = V_R3 / I_R3
print(f"Calculated value of R3: {R3:.2f} ohms")

V_R5 = Vcc / 4 # Voltage across R5 (quarter of the supply voltage for proper biasing)
R5 = V_R5 / Ie_Q1
print(f"Calculated value of R5: {R5:.2f} ohms")

R1 = 100 # Keep R1 value the same as on the original schematic
Ve_Q1 = Ie_Q1 * (R1 + R5) # Voltage at the emitter of Q1
Vb_Q1 = Ve_Q1 + 0.7 # Voltage at the base of Q1 (0.7V above the emitter voltage)
I_R6 = I_R2 - Ib_Q1 # Current through R6 (current through R2 minus base current of Q1)
R6 = Vb_Q1 / I_R6
print(f"Calculated value of R6: {R6:.2f} ohms")

R2 = (Vcc/2 - Vb_Q1) / I_R2
print(f"Calculated value of R2: {R2:.2f} ohms")

Calculated value of R3: 2272.73 ohms
Calculated value of R5: 1250.00 ohms
Calculated value of R6: 22777.78 ohms
Calculated value of R2: 4500.00 ohms


The component values are:
| Component | Value |
| --- | --- |
| R2 | 4.7kΩ |
| R3 | 2.2kΩ |
| R5 | 1.2kΩ |
| R6 | 22kΩ |


# Class A amplifier
EMRFD fig. 2.34 shows a common-emitter amplifier with an 8.2mA emitter current. The emitter sits at about 1/6 of the supply voltage.

In [12]:
import math

Vcc = 5 # Supply voltage
V_E = 0.8 # Voltage at the emitter of Q1
Ie_Q1 = 8e-3 # Emitter current of Q1
R9 = V_E / Ie_Q1
print(f"Calculated value of R9: {R9:.2f} ohms")

Vb_Q1 = V_E + 0.7 # Voltage at the base of Q1 (0.7V above the emitter voltage)
f_T = 300e6 # Transition frequency of the transistor (300 MHz)
hfe_DC = 300
f = 10e6 # Frequency of interest (10 MHz)
hfe_10MHz = min(hfe_DC, f_T / f)
Ib_Q1 = Ie_Q1 / hfe_10MHz
I_R8 = 10 * Ib_Q1 # Current through R8 (10 times the base current of Q1 for good biasing)
R8 = Vb_Q1 / I_R8
print(f"Calculated value of R8: {R8:.2f} ohms")

R7 = (Vcc - Vb_Q1) / (Ib_Q1 + I_R8)
print(f"Calculated value of R7: {R7:.2f} ohms")

# Calculate the inductor impedance
L = 2*6.8e-6 # Inductance of L1 (2 times 6.8 uH)
Z_L = 2 * math.pi * f * L
print(f"Calculated impedance of L1 at 10 MHz: {Z_L:.2f} ohms")

# Amplifier gain calculation
R10 = 1.5e3
V_T = 0.025 # Thermal voltage at room temperature (25 mV)
r_e = V_T / Ie_Q1 # Intrinsic emitter resistance
Z_out = Z_L * R10 / (Z_L + R10) # Output impedance of the amplifier (ignoring NMOS input impedance)
A_v = -Z_out / r_e # Voltage gain of the amplifier
print(f"Calculated voltage gain of the amplifier at 10 MHz: {A_v:.2f}")

Calculated value of R9: 100.00 ohms
Calculated value of R8: 562.50 ohms
Calculated value of R7: 1193.18 ohms
Calculated impedance of L1 at 10 MHz: 854.51 ohms
Calculated voltage gain of the amplifier at 10 MHz: -174.20


The value of R7 is set to 1.5k instead of 1.2k to reduce the emitter current to about 6.5mA.  The inductor value has been reduced to 2*6.8µH.
In simulation, the Vpp on the oscillator output is 53mV.  The Vpp output on the amplifier is 1.64V, which is a gain of about 31.  